In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("NYC_Taxi_Data_Pipeline")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "12")
    .getOrCreate()
)

In [3]:
spark.range(10).count()

10

In [5]:
df_raw_2024 = spark.read.parquet(
    "../data/raw/yellow_tripdata_2024-01.parquet",
    "../data/raw/yellow_tripdata_2024-02.parquet",
    "../data/raw/yellow_tripdata_2024-03.parquet"
)

In [6]:
df_raw_2024.explain()

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [VendorID#25,tpep_pickup_datetime#26,tpep_dropoff_datetime#27,passenger_count#28L,trip_distance#29,RatecodeID#30L,store_and_fwd_flag#31,PULocationID#32,DOLocationID#33,payment_type#34L,fare_amount#35,extra#36,mta_tax#37,tip_amount#38,tolls_amount#39,improvement_surcharge#40,total_amount#41,congestion_surcharge#42,Airport_fee#43] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(3 paths)[file:/C:/Users/Murch24/aws-pyspark-data-engineering-project/data/raw/y..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<VendorID:int,tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,passen...




In [7]:
df_raw_2024.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [8]:
df_raw_2024.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:57:55|  2024-01-01 01:17:43|              1|         1.72|         1|                 N|         186|          79|           2|       17.7|  1.0|    0.5|       0.

In [9]:
df_raw_2024.count()

9554778

## Validate Date Range

Verify that all records belong to the expected time period before further processing.

In [10]:
df_raw_2024.select(
    F.min("tpep_pickup_datetime"),
    F.max("tpep_pickup_datetime")
).show()

+-------------------------+-------------------------+
|min(tpep_pickup_datetime)|max(tpep_pickup_datetime)|
+-------------------------+-------------------------+
|      2002-12-31 22:17:10|      2024-04-01 00:34:55|
+-------------------------+-------------------------+



## Filter Invalid Years

A small number of records were found outside the expected year (2024). These records are treated as data quality issues and removed from the analytical dataset.

In [11]:
df_raw_2024.select(
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
    "store_and_fwd_flag"
).describe().show()

+-------+------------------+------------------+------------------+------------------+--------------------+------------------+------------------+
|summary|     trip_distance|       fare_amount|        tip_amount|      total_amount|congestion_surcharge|       airport_fee|store_and_fwd_flag|
+-------+------------------+------------------+------------------+------------------+--------------------+------------------+------------------+
|  count|           9554778|           9554778|           9554778|           9554778|             8802816|           8802816|           8802816|
|   mean| 4.042286145214355|18.325116607624924|3.2710800240464804|26.865397974733813|  2.2580984028292765|0.1377972401104374|              NULL|
| stddev|265.47827357783393| 18.54496529152651| 3.927615572116735|23.050191012603392|  0.8254055232053454|0.4831946695622556|              NULL|
|    min|               0.0|            -999.0|            -300.0|           -1000.0|                -2.5|             -1.75|     

In [12]:
len(df_raw_2024.columns)

19

In [13]:
df_raw_2024.rdd.getNumPartitions()

12

#### I usually start with data profiling using describe() or summary(). I check null counts, distributions, percentiles, and extreme values before applying data quality rules. For example, mean and max comparison can reveal outliers that may affect analytics.

In [14]:
df_raw_2024.select(
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
    "store_and_fwd_flag"
).summary().show()

+-------+------------------+------------------+------------------+------------------+--------------------+------------------+------------------+
|summary|     trip_distance|       fare_amount|        tip_amount|      total_amount|congestion_surcharge|       airport_fee|store_and_fwd_flag|
+-------+------------------+------------------+------------------+------------------+--------------------+------------------+------------------+
|  count|           9554778|           9554778|           9554778|           9554778|             8802816|           8802816|           8802816|
|   mean| 4.042286145214355|18.325116607624924|3.2710800240464804|26.865397974733813|  2.2580984028292765|0.1377972401104374|              NULL|
| stddev|265.47827357783393| 18.54496529152651| 3.927615572116735|23.050191012603392|  0.8254055232053454|0.4831946695622556|              NULL|
|    min|               0.0|            -999.0|            -300.0|           -1000.0|                -2.5|             -1.75|     

In [15]:
null_counts = df_raw_2024.select(
    [
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df_raw_2024.columns
    ]
)

null_counts.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       0|                   0|                    0|         751962|            0|    751962|            751962|           0|           0|           0|          0|    0|      0|         

In [16]:
df_raw_2024.filter(
    F.col("passenger_count").isNull()
    &
    F.col("RatecodeID").isNull()
    &
    F.col("store_and_fwd_flag").isNull()
    &
    F.col("congestion_surcharge").isNull()
    &
    F.col("Airport_fee").isNull()
).count()


751962

Finding:
751,962 records (7.87%) contain NULL values in:
- passenger_count
- RatecodeID
- store_and_fwd_flag
- congestion_surcharge
- Airport_fee

The NULL values are not random. All affected records have payment_type = 0
and are mainly associated with VendorID 1 and 2.

Core trip attributes (pickup/dropoff time, distance, fare_amount, total_amount,
location IDs) are populated, therefore records are retained.

Decision:
- Keep records.
- Do not impute categorical fields.
- Investigate surcharge fields before replacing NULL with zero.

In [17]:
df_raw_2024.filter(
    F.col("passenger_count").isNull()
).select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "PULocationID",
    "DOLocationID"
).show()

+--------------------+---------------------+-------------+-----------+------------+------------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|total_amount|PULocationID|DOLocationID|
+--------------------+---------------------+-------------+-----------+------------+------------+------------+
| 2024-01-01 00:34:19|  2024-01-01 00:51:22|         2.04|      12.72|       16.72|         143|         141|
| 2024-01-01 00:14:31|  2024-01-01 00:19:29|          1.6|        9.3|       17.16|         236|         238|
| 2024-01-01 00:35:11|  2024-01-01 01:13:40|          0.0|      21.01|       25.01|         142|          79|
| 2024-01-01 00:33:37|  2024-01-01 00:50:34|          0.0|      17.79|       21.79|         237|           4|
| 2024-01-01 00:49:04|  2024-01-01 01:01:16|          0.0|      34.65|       38.65|         244|          50|
| 2024-01-01 00:47:00|  2024-01-01 01:11:05|         4.58|      28.14|       29.64|         202|          83|
| 2024-01-

In [18]:
df_raw_2024.filter(
    F.col("passenger_count").isNull() 
).groupBy(
    "VendorID"
).count().show()

+--------+------+
|VendorID| count|
+--------+------+
|       6|   720|
|       2|542016|
|       1|209226|
+--------+------+



In [19]:
df_raw_2024.filter(
    F.col("passenger_count").isNull()
).groupBy(
    "payment_type"
).count().show()

+------------+------+
|payment_type| count|
+------------+------+
|           0|751962|
+------------+------+



In [20]:
df_raw_2024.filter(
    F.col("passenger_count").isNotNull()
).groupBy(
    "payment_type"
).count().show()

+------------+-------+
|payment_type|  count|
+------------+-------+
|           3|  61966|
|           1|7258158|
|           2|1330105|
|           4| 152587|
+------------+-------+



In [21]:
df_raw_2024.filter(
    F.col("congestion_surcharge").isNull()
).select(
    "congestion_surcharge",
    "fare_amount",
    "total_amount",
    "tip_amount",
    "tolls_amount"
).summary().show()

+-------+--------------------+-----------------+------------------+------------------+------------------+
|summary|congestion_surcharge|      fare_amount|      total_amount|        tip_amount|      tolls_amount|
+-------+--------------------+-----------------+------------------+------------------+------------------+
|  count|                   0|           751962|            751962|            751962|            751962|
|   mean|                NULL|19.00612659948637|24.017144802532485|0.9169331827937883|0.2379332599253881|
| stddev|                NULL|16.51345879176269|17.953398053935484|2.3769724481659305|1.4323519419652138|
|    min|                NULL|          -120.52|            -99.15|               0.0|               0.0|
|    25%|                NULL|            11.62|             15.87|               0.0|               0.0|
|    50%|                NULL|            16.63|              21.2|               0.0|               0.0|
|    75%|                NULL|            24.2

In [22]:
df_raw_2024.filter(
    F.col("congestion_surcharge").isNotNull()
).select(
    "congestion_surcharge",
    "fare_amount",
    "total_amount",
    "tip_amount",
    "tolls_amount"
).summary().show()

+-------+--------------------+------------------+------------------+------------------+-----------------+
|summary|congestion_surcharge|       fare_amount|      total_amount|        tip_amount|     tolls_amount|
+-------+--------------------+------------------+------------------+------------------+-----------------+
|  count|             8802816|           8802816|           8802816|           8802816|          8802816|
|   mean|  2.2580984028292765|18.266942764680095|27.108703997708155|3.4721780552949078|0.550993855829565|
| stddev|  0.8254055232053454| 18.70712985792123|23.418185379808026| 3.968301705353132|2.171661788527764|
|    min|                -2.5|            -999.0|           -1000.0|            -300.0|            -84.3|
|    25%|                 2.5|               8.6|              15.4|               1.0|              0.0|
|    50%|                 2.5|              12.8|             20.16|               2.8|              0.0|
|    75%|                 2.5|              20

In [23]:
df_zones = spark.read.csv(
    "../data/raw/taxi_zone_lookup.csv",
    header=True,
    inferSchema=True
)

In [24]:
from pyspark.sql.functions import broadcast

airport_null = df_raw_2024.filter(
    F.col("Airport_fee").isNull()
).groupBy(
    "PULocationID"
).count()

airport_null.join(
        broadcast(df_zones),
        airport_null.PULocationID == df_zones.LocationID,
        "left"
    ).filter(
        F.col("Zone").isin(
            "JFK Airport",
            "LaGuardia Airport",
            "Newark Airport"
        )
    ).show()


+------------+-----+----------+-------+-----------------+------------+
|PULocationID|count|LocationID|Borough|             Zone|service_zone|
+------------+-----+----------+-------+-----------------+------------+
|         132| 1184|       132| Queens|      JFK Airport|    Airports|
|         138| 1610|       138| Queens|LaGuardia Airport|    Airports|
|           1|    4|         1|    EWR|   Newark Airport|         EWR|
+------------+-----+----------+-------+-----------------+------------+



##### I don't remove outliers based only on one column. I validate them using related business attributes. For taxi data, a high trip distance may be valid, so I also check duration and calculated speed before filtering.

In [25]:
df_raw_2024.withColumn(
    "duration_hours",
    (F.unix_timestamp("tpep_dropoff_datetime") -
     F.unix_timestamp("tpep_pickup_datetime")) / 3600
).withColumn(
    "speed",
    F.col("trip_distance") / F.col("duration_hours")
).filter(
    F.col("trip_distance") > 100
).select(
    "trip_distance",
    "speed",
    "fare_amount"
).show(20)

+-------------+------------------+-----------+
|trip_distance|             speed|fare_amount|
+-------------+------------------+-----------+
|       101.28|50.457791309161365|      359.3|
|       233.25| 63.80214269432414|     1616.5|
|        971.8|  646.430155210643|       21.5|
|        964.6| 3919.367945823928|       39.5|
|     10879.28|14624.872292755788|       70.0|
|       142.62| 53.44909431605247|      912.3|
|       111.57| 58.52426052746612|      678.5|
|        101.1| 39.38961038961038|      450.0|
|       115.75| 62.13838353713093|      400.0|
|       176.43| 457.9293439077145|       35.9|
|       210.82|  67.8362531283518|      500.0|
|       106.57|  43.0537537874537|      366.0|
|       120.78| 47.59281961471103|      325.0|
|       120.86|48.194062915374396|      457.0|
|        108.6| 54.78699551569506|      380.0|
|       122.47| 56.08599414832719|      749.2|
|      1715.22|2428.1525756979945|       70.0|
|       135.82| 69.92020592020592|      220.0|
|        116.

In [26]:
df_2024_clean = (
    df_raw_2024
    .withColumn(
        "passenger_count_missing",
        F.when(F.col("passenger_count").isNull(), 1).otherwise(0)
    )
    .withColumn(
        "congestion_surcharge_missing",
        F.when(F.col("congestion_surcharge").isNull(), 1).otherwise(0)
    )
    .withColumn(
        "Airport_fee_missing",
        F.when(F.col("Airport_fee").isNull() & F.col("PULocationID").isin(
            132,1,138), 1).otherwise(0)
    )
)    

In [27]:
df_2024_clean.select(
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "passenger_count",
    "passenger_count_missing",
    "congestion_surcharge",
    "congestion_surcharge_missing",
    "Airport_fee",
    "PULocationID",
    "Airport_fee_missing",
    "store_and_fwd_flag"
).filter(
    F.col("Airport_fee").isNull()
).show()

+-------------+-----------+----------+------------+---------------+-----------------------+--------------------+----------------------------+-----------+------------+-------------------+------------------+
|trip_distance|fare_amount|tip_amount|total_amount|passenger_count|passenger_count_missing|congestion_surcharge|congestion_surcharge_missing|Airport_fee|PULocationID|Airport_fee_missing|store_and_fwd_flag|
+-------------+-----------+----------+------------+---------------+-----------------------+--------------------+----------------------------+-----------+------------+-------------------+------------------+
|         2.04|      12.72|       0.0|       16.72|           NULL|                      1|                NULL|                           1|       NULL|         143|                  0|              NULL|
|          1.6|        9.3|      2.86|       17.16|           NULL|                      1|                NULL|                           1|       NULL|         236|          

In [28]:
df_raw_2024.filter(
    F.col("Airport_fee").isNull()
).join(
    broadcast(df_zones),
    F.col("PULocationID") == F.col("LocationID"),
    "left"
).filter(
    F.col("Zone").isin(
        "JFK Airport",
        "LaGuardia Airport",
        "Newark Airport"
    )
).select(
    "PULocationID",
    "Zone",
    "fare_amount",
    "total_amount",
    "airport_fee",
    "payment_type",
    "VendorID"
).show(20)

+------------+-----------------+-----------+------------+-----------+------------+--------+
|PULocationID|             Zone|fare_amount|total_amount|airport_fee|payment_type|VendorID|
+------------+-----------------+-----------+------------+-----------+------------+--------+
|         132|      JFK Airport|      60.61|       71.55|       NULL|           0|       2|
|         132|      JFK Airport|       70.0|       95.09|       NULL|           0|       1|
|         132|      JFK Airport|       70.0|       90.69|       NULL|           0|       1|
|         138|LaGuardia Airport|       31.0|       58.42|       NULL|           0|       1|
|         138|LaGuardia Airport|       33.8|       59.21|       NULL|           0|       1|
|         138|LaGuardia Airport|       44.3|       66.06|       NULL|           0|       1|
|         132|      JFK Airport|       71.6|       86.07|       NULL|           0|       1|
|         138|LaGuardia Airport|       51.3|       75.88|       NULL|           

In [29]:
df_raw_2024.filter(
    F.col("passenger_count") == 0
).select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "payment_type",
    "VendorID"
).show(20)

+--------------------+---------------------+-------------+-----------+------------+------------+--------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|total_amount|payment_type|VendorID|
+--------------------+---------------------+-------------+-----------+------------+------------+--------+
| 2024-01-01 00:30:40|  2024-01-01 00:58:40|          3.0|       25.4|        30.4|           2|       1|
| 2024-01-01 00:49:42|  2024-01-01 00:58:21|          2.9|       14.9|        24.9|           1|       1|
| 2024-01-01 00:42:26|  2024-01-01 01:12:01|         16.6|       70.0|       99.22|           1|       1|
| 2024-01-01 00:38:18|  2024-01-01 01:01:12|          2.5|       17.0|        27.5|           1|       1|
| 2024-01-01 00:26:21|  2024-01-01 00:38:03|          3.4|       15.6|       22.66|           1|       1|
| 2024-01-01 00:38:43|  2024-01-01 00:51:53|          2.6|       14.9|       23.85|           1|       1|
| 2024-01-01 00:19:40|  2024-01-01 00:38:32|  

In [30]:
df_raw_2024.filter(
    F.col("passenger_count") == 0
).groupBy(
    "VendorID"
).count().show()

+--------+------+
|VendorID| count|
+--------+------+
|       2|   374|
|       1|105557|
+--------+------+



In [31]:
df_raw_2024.filter(
    F.col("passenger_count") == 0
).groupBy(
    F.month("tpep_pickup_datetime")
).count().show()

+---------------------------+-----+
|month(tpep_pickup_datetime)|count|
+---------------------------+-----+
|                          1|31465|
|                          2|34094|
|                          3|40372|
+---------------------------+-----+



In [32]:
df_raw_2024.groupBy(
    "VendorID",
    "passenger_count"
).count().orderBy(
    "VendorID",
    "passenger_count"
).show(50)

+--------+---------------+-------+
|VendorID|passenger_count|  count|
+--------+---------------+-------+
|       1|           NULL| 209226|
|       1|              0| 105557|
|       1|              1|1652125|
|       1|              2| 275007|
|       1|              3|  58676|
|       1|              4|  29945|
|       1|              5|   1491|
|       1|              6|   2196|
|       1|              8|      1|
|       1|              9|      2|
|       2|           NULL| 542016|
|       2|              0|    374|
|       2|              1|5167689|
|       2|              2| 987666|
|       2|              3| 227307|
|       2|              4| 134645|
|       2|              5|  96555|
|       2|              6|  63484|
|       2|              7|     14|
|       2|              8|     76|
|       2|              9|      6|
|       6|           NULL|    720|
+--------+---------------+-------+



In [33]:
df_raw_2024.filter(
    F.col("passenger_count") == 0
).join(
    broadcast(df_zones),
    F.col("PULocationID") == F.col("LocationID"),
    "left"
).groupBy(
    "zone"
).count().orderBy(
    F.desc("count")
).show()

+--------------------+-----+
|                zone|count|
+--------------------+-----+
|      Midtown Center| 5412|
|Upper East Side S...| 5166|
|Upper East Side N...| 4629|
| Lincoln Square East| 4200|
|Penn Station/Madi...| 4055|
|        Midtown East| 3714|
|Times Sq/Theatre ...| 3585|
|Upper West Side S...| 3449|
|       Midtown North| 3345|
|         JFK Airport| 3095|
|         Murray Hill| 3021|
|            Union Sq| 2954|
|        East Chelsea| 2801|
|        Clinton East| 2658|
|     Lenox Hill West| 2566|
|     Lenox Hill East| 2557|
|        East Village| 2466|
|   LaGuardia Airport| 2409|
|        West Village| 2408|
|       Midtown South| 2390|
+--------------------+-----+
only showing top 20 rows


In [34]:
df_raw_2024.filter(
    F.col("passenger_count") == 0
).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2024-01-01 00:30:40|  2024-01-01 00:58:40|              0|          3.0|         1|                 N|         246|         231|           2|       25.4|  3.5|    0.5|       0.

### Negative amounts (`fare_amount`, `total_amount`, `tip_amount`, `tolls_amount`)

In [35]:
negative_amounts = df_raw_2024.filter(
    (F.col("fare_amount") < 0) |
    (F.col("total_amount") < 0) |
    (F.col("tip_amount") < 0) |
    (F.col("tolls_amount") < 0)
)

negative_amounts.count()

136908

In [36]:
df_raw_2024.filter(
    F.col("total_amount") < 0
).count()

115895

In [37]:
df_raw_2024.filter(
    F.col("fare_amount") < 0
).count()

136567

In [38]:
df_raw_2024.filter(
    F.col("tip_amount") < 0
).count()

330

In [39]:
df_raw_2024.filter(
    F.col("tolls_amount") < 0
).count()

7586

In [40]:
df_raw_2024.filter(
    (F.col("total_amount") < 0) 
).select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "tip_amount",
    "tolls_amount",
    "payment_type",
    "VendorID"
).show()

+--------------------+---------------------+-------------+-----------+------------+----------+------------+------------+--------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|total_amount|tip_amount|tolls_amount|payment_type|VendorID|
+--------------------+---------------------+-------------+-----------+------------+----------+------------+------------+--------+
| 2024-01-01 00:18:24|  2024-01-01 00:30:39|         2.16|      -13.5|       -18.5|       0.0|         0.0|           4|       2|
| 2024-01-01 00:04:00|  2024-01-01 00:04:44|         0.01|      -31.5|      -34.25|       0.0|         0.0|           2|       2|
| 2024-01-01 00:41:42|  2024-01-01 00:46:00|         0.47|       -5.8|       -10.8|       0.0|         0.0|           4|       2|
| 2024-01-01 00:42:02|  2024-01-01 01:14:33|         5.48|      -33.1|       -38.1|       0.0|         0.0|           2|       2|
| 2024-01-01 00:24:02|  2024-01-01 01:10:32|         8.74|      -47.8|       -52.8|       

In [41]:
negative_amounts.groupBy(
    "VendorID"
).count().show()

+--------+------+
|VendorID| count|
+--------+------+
|       2|136908|
+--------+------+



In [42]:
negative_amounts.groupBy(
    "payment_type"
).count().show()

+------------+-----+
|payment_type|count|
+------------+-----+
|           3|18284|
|           1|   87|
|           4|69970|
|           2|27543|
|           0|21024|
+------------+-----+



In [43]:
negative_amounts.groupBy(
    F.month("tpep_pickup_datetime")
).count().show()

+---------------------------+-----+
|month(tpep_pickup_datetime)|count|
+---------------------------+-----+
|                          1|37568|
|                         12|    1|
|                          2|40733|
|                          3|58606|
+---------------------------+-----+



### During data quality analysis, 136,908 records with negative monetary values were identified. All affected records originated from VendorID=2. The records were distributed across multiple months and payment types, indicating a systematic source-specific pattern rather than a loading error. A significant portion of these records (52,785) had trip distances below 1 mile, and examples showed zero passengers, near-zero distances, and negative fare/total amounts. These records appear to represent non-standard transactions (such as adjustments or reversals) rather than normal completed taxi trips.
Среди 136908 записей с отрицательными суммами, 20 записей с passenger_count=0


In [44]:
negative_amounts.filter(
    (F.col("passenger_count")== 0) 
).count()

20

In [45]:
negative_amounts.filter(
    (F.col("passenger_count")== 0) 
).select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "tip_amount",
    "tolls_amount",
    "payment_type",
    "VendorID"
).show()

+--------------------+---------------------+-------------+-----------+------------+----------+------------+------------+--------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|total_amount|tip_amount|tolls_amount|payment_type|VendorID|
+--------------------+---------------------+-------------+-----------+------------+----------+------------+------------+--------+
| 2024-01-17 23:30:34|  2024-01-17 23:31:21|          0.0|       -8.2|       -12.2|       0.0|         0.0|           4|       2|
| 2024-01-23 12:59:18|  2024-01-23 12:59:33|          0.0|       -6.5|        -8.0|       0.0|         0.0|           4|       2|
| 2024-01-26 15:02:38|  2024-01-26 15:02:43|          0.0|      -31.7|       -35.7|       0.0|         0.0|           3|       2|
| 2024-01-26 21:27:54|  2024-01-26 21:28:02|         0.02|       -7.5|       -11.5|       0.0|         0.0|           3|       2|
| 2024-01-30 05:51:58|  2024-01-30 05:52:09|         0.04|      -23.0|       -25.0|       

In [46]:
negative_amounts.filter(
    (F.col("trip_distance")<1) 
).select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "tip_amount",
    "tolls_amount",
    "payment_type",
    "VendorID"
).count()

52785

In [47]:
negative_amounts.filter(
    (F.col("trip_distance")<1) 
).select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "tip_amount",
    "tolls_amount",
    "payment_type",
    "VendorID"
).show(20)

+--------------------+---------------------+-------------+-----------+------------+----------+------------+------------+--------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|total_amount|tip_amount|tolls_amount|payment_type|VendorID|
+--------------------+---------------------+-------------+-----------+------------+----------+------------+------------+--------+
| 2024-01-01 00:04:00|  2024-01-01 00:04:44|         0.01|      -31.5|      -34.25|       0.0|         0.0|           2|       2|
| 2024-01-01 00:41:42|  2024-01-01 00:46:00|         0.47|       -5.8|       -10.8|       0.0|         0.0|           4|       2|
| 2024-01-01 00:33:28|  2024-01-01 00:33:57|          0.0|       -3.0|        -8.0|       0.0|         0.0|           4|       2|
| 2024-01-01 00:14:14|  2024-01-01 00:24:57|         0.63|      -10.0|       -15.0|       0.0|         0.0|           2|       2|
| 2024-01-01 00:17:55|  2024-01-01 00:18:05|         0.01|      -70.0|       -74.0|       

In [48]:
negative_amounts.filter(
    (F.col("trip_distance")<1) 
).groupBy(
    "VendorID"
).count().show()

+--------+-----+
|VendorID|count|
+--------+-----+
|       2|52785|
+--------+-----+



In [49]:
df_2024_clean.count()

9554778

In [50]:
df_2024_clean = (
    df_2024_clean
    .withColumn(
        "negative_amount_flag",
        F.when(
            (F.col("fare_amount") < 0) |
            (F.col("total_amount") < 0) |
            (F.col("tip_amount") < 0) |
            (F.col("tolls_amount") < 0),
            1
        ).otherwise(0)
    )
)

In [51]:
df_2024_clean.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- passenger_count_missing: integer (nullable = false)
 |-- congestion_surcharge_missing: integer (nullable = false)
 |-- Ai

In [52]:
df_raw_2024.filter(
    F.col("trip_distance") > 100
).count()

252

In [53]:
df_raw_2024.filter(
    F.col("trip_distance") > 100
).select(
    "trip_distance",
    "fare_amount",
    "total_amount",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime"
).show(20)

+-------------+-----------+------------+--------------------+---------------------+
|trip_distance|fare_amount|total_amount|tpep_pickup_datetime|tpep_dropoff_datetime|
+-------------+-----------+------------+--------------------+---------------------+
|       101.28|      359.3|       364.3| 2024-01-02 02:38:04|  2024-01-02 04:38:30|
|       233.25|     1616.5|      1617.5| 2024-01-02 07:50:08|  2024-01-02 11:29:29|
|        971.8|       21.5|        23.0| 2024-01-02 08:13:18|  2024-01-02 09:43:30|
|        964.6|       39.5|        41.0| 2024-01-02 08:40:41|  2024-01-02 08:55:27|
|     10879.28|       70.0|       98.88| 2024-01-03 11:02:57|  2024-01-03 11:47:35|
|       142.62|      912.3|      940.93| 2024-01-06 21:01:38|  2024-01-06 23:41:44|
|       111.57|      678.5|       696.0| 2024-01-07 19:27:28|  2024-01-07 21:21:51|
|        101.1|      450.0|      551.59| 2024-01-09 14:14:49|  2024-01-09 16:48:49|
|       115.75|      400.0|      409.69| 2024-01-10 11:14:43|  2024-01-10 13

In [54]:
df_raw_2024.select(
    "trip_distance"
).summary().show()

+-------+------------------+
|summary|     trip_distance|
+-------+------------------+
|  count|           9554778|
|   mean| 4.042286145214355|
| stddev|265.47827357783393|
|    min|               0.0|
|    25%|               1.0|
|    50%|               1.7|
|    75%|              3.19|
|    max|          312722.3|
+-------+------------------+



In [55]:
long_distance_check = df_2024_clean.filter(
    F.col("trip_distance") > 100
).withColumn(
        "trip_duration_minutes",
        (F.unix_timestamp("tpep_dropoff_datetime") -
         F.unix_timestamp("tpep_pickup_datetime")) / 60
    ).withColumn(
        "avg_speed_mph",
        F.when(
            F.col("trip_duration_minutes") > 0,
            F.col("trip_distance") /
            (F.col("trip_duration_minutes") / 60)
        )
    )


In [56]:
long_distance_check.select(
   "trip_distance",
   "trip_duration_minutes",
    "avg_speed_mph"
).orderBy(F.desc("avg_speed_mph")
).show()

+-------------+---------------------+------------------+
|trip_distance|trip_duration_minutes|     avg_speed_mph|
+-------------+---------------------+------------------+
|    222478.29|                  7.0|1906956.7714285715|
|     312722.3|                 13.0|1443333.6923076923|
|     59076.43|                  4.0| 886146.4500000001|
|    172935.77|                 12.0| 864678.8499999999|
|     176836.3|                 13.0| 816167.5384615384|
|     136660.1|                 12.0|          683300.5|
|    105744.27|                 10.0| 634465.6200000001|
|     64674.26|                  7.0|          554350.8|
|    176329.23|                 20.0| 528987.6900000001|
|     51657.73|                  6.0|          516577.3|
|    154959.09|                 18.0|          516530.3|
|     77456.58|                  9.0|          516377.2|
|     65494.94|                  9.0|436632.93333333335|
|      47224.4|                  7.0| 404780.5714285714|
|     66838.02|                

In [57]:
long_distance_check.select(
    "trip_distance",
    "trip_duration_minutes",
    "avg_speed_mph"
).orderBy(
    F.col("avg_speed_mph").desc()
).show()

+-------------+---------------------+------------------+
|trip_distance|trip_duration_minutes|     avg_speed_mph|
+-------------+---------------------+------------------+
|    222478.29|                  7.0|1906956.7714285715|
|     312722.3|                 13.0|1443333.6923076923|
|     59076.43|                  4.0| 886146.4500000001|
|    172935.77|                 12.0| 864678.8499999999|
|     176836.3|                 13.0| 816167.5384615384|
|     136660.1|                 12.0|          683300.5|
|    105744.27|                 10.0| 634465.6200000001|
|     64674.26|                  7.0|          554350.8|
|    176329.23|                 20.0| 528987.6900000001|
|     51657.73|                  6.0|          516577.3|
|    154959.09|                 18.0|          516530.3|
|     77456.58|                  9.0|          516377.2|
|     65494.94|                  9.0|436632.93333333335|
|      47224.4|                  7.0| 404780.5714285714|
|     66838.02|                

In [58]:
df_2024_clean = df_2024_clean.withColumn(
        "trip_duration_minutes",
        (F.unix_timestamp("tpep_dropoff_datetime") -
         F.unix_timestamp("tpep_pickup_datetime")) / 60
    ).withColumn(
        "avg_speed_mph",
        F.when(
            F.col("trip_duration_minutes") > 0,
            F.col("trip_distance") /
            (F.col("trip_duration_minutes") / 60)
        )
)   

In [59]:
df_2024_clean.filter(
    F.col("avg_speed_mph").isNull()
).count()

2801

### Records with unrealistic average speeds were identified among long-distance trips. These records were flagged as potential distance anomalies and retained for traceability.

In [60]:
df_2024_clean = df_2024_clean.withColumn(
    "long_distance_flag",
    F.when(
        (F.col("trip_distance") > 100) &
        (F.col("avg_speed_mph") > 85),
        1
    ).otherwise(0)
)

In [61]:
df_2024_clean.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- passenger_count_missing: integer (nullable = false)
 |-- congestion_surcharge_missing: integer (nullable = false)
 |-- Ai

In [62]:
df_2024_clean.groupBy(
    "long_distance_flag"
).count().show()

+------------------+-------+
|long_distance_flag|  count|
+------------------+-------+
|                 0|9554608|
|                 1|    170|
+------------------+-------+



 ### `trip_duration_minutes` and `pickup_date` дают возможность делать аналитику по:

- длительности поездок;
- дням;
- периодам;
- трендам.

In [63]:
df_2024_clean = df_2024_clean.withColumn(
    "pickup_date",
    F.to_date("tpep_pickup_datetime")
)

#### `pickup_hour`, `pickup_day_of_week`, and `avg_speed_mph` 
 ### дают возможность делать аналитику по:

- в какие часы больше всего поездок?;
- анализ будни/выходные;
- скорость поездок.

In [64]:
df_2024_clean = df_2024_clean.withColumn(
    "pickup_hour",
    F.hour("tpep_pickup_datetime")
)

In [65]:
df_2024_clean = df_2024_clean.withColumn(
    "pickup_day_of_week",
    F.dayofweek("tpep_pickup_datetime")
)

In [66]:
df_2024_clean.select(
    "tpep_pickup_datetime",
    "pickup_date",
    "pickup_hour",
    "pickup_day_of_week"
).show(5)

+--------------------+-----------+-----------+------------------+
|tpep_pickup_datetime|pickup_date|pickup_hour|pickup_day_of_week|
+--------------------+-----------+-----------+------------------+
| 2024-01-01 00:57:55| 2024-01-01|          0|                 2|
| 2024-01-01 00:03:00| 2024-01-01|          0|                 2|
| 2024-01-01 00:17:06| 2024-01-01|          0|                 2|
| 2024-01-01 00:36:38| 2024-01-01|          0|                 2|
| 2024-01-01 00:46:51| 2024-01-01|          0|                 2|
+--------------------+-----------+-----------+------------------+
only showing top 5 rows


### Select Analytical Columns

We select the columns needed for analysis.
The analytical dataset contains original business fields and newly created features.

In [67]:
df_2024_analytics = df_2024_clean.filter(
    (F.col("negative_amount_flag") == 0) &
    (F.col("long_distance_flag") == 0) &
    (F.year("pickup_date") == 2024) &
    (F.month("pickup_date").isin(1,2,3))
).select(
    "VendorID",
    "pickup_date",
    "pickup_hour",
    "pickup_day_of_week",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "trip_duration_minutes",
    "avg_speed_mph",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "payment_type",
    "store_and_fwd_flag"
)

In [68]:
df_2024_analytics.select(
    F.min("pickup_date"),
    F.max("pickup_date")
).show()

+----------------+----------------+
|min(pickup_date)|max(pickup_date)|
+----------------+----------------+
|      2024-01-01|      2024-03-31|
+----------------+----------------+



In [69]:
df_2024_analytics.count()

9417683

In [70]:
df_2024_analytics.filter(
    (F.col("fare_amount") < 0) |
    (F.col("total_amount") < 0) |
    (F.col("tip_amount") < 0) |
    (F.col("tolls_amount") < 0)
).count()

0

In [71]:
df_2024_analytics.filter(
    (F.col("trip_distance") > 100) &
    (F.col("avg_speed_mph") > 85)
).count()

0

```
df_raw_2024
      |
      |  data quality checks
      |  + derived columns
      |  + DQ flags
      ↓
df_2024_clean
      |
      |  apply business rules
      |  remove records with:
      |     negative_amount_flag = 1
      |     long_distance_flag = 1
      ↓
df_2024_analytics
```

In [72]:
df_2024_analytics.select(
    [
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df_2024_analytics.columns
    ]
).show()

+--------+-----------+-----------+------------------+------------+------------+---------------+-------------+---------------------+-------------+-----------+----------+------------+------------+------------+------------------+
|VendorID|pickup_date|pickup_hour|pickup_day_of_week|PULocationID|DOLocationID|passenger_count|trip_distance|trip_duration_minutes|avg_speed_mph|fare_amount|tip_amount|tolls_amount|total_amount|payment_type|store_and_fwd_flag|
+--------+-----------+-----------+------------------+------------+------------+---------------+-------------+---------------------+-------------+-----------+----------+------------+------------+------------+------------------+
|       0|          0|          0|                 0|           0|           0|         730821|            0|                    0|         2790|          0|         0|           0|           0|           0|            730821|
+--------+-----------+-----------+------------------+------------+------------+-------------

### Data Quality validation - df_2024_analytics

Null value analysis after applying business rules:

| Column | NULL count | Decision |
|---|---:|---|
| passenger_count | 730,821 | Keep - missing passenger information does not affect financial metrics |
| store_and_fwd_flag | 730,821 | Keep - optional operational field |
| avg_speed_mph | 2,790 | Keep - derived metric, NULL values do not affect core financial analysis |

All critical analytical fields have no NULL values:
- trip_distance
- trip_duration_minutes
- fare_amount
- tip_amount
- tolls_amount
- total_amount
- payment_type
- pickup/dropoff locations

The dataset is ready for analytical processing.

In [73]:
df_2024_analytics.rdd.getNumPartitions()

12

### Spark Internals: Lazy Evaluation and Explain Plan

Spark uses lazy evaluation. Transformations are not executed immediately.
Spark builds an execution plan and runs the computation only when an action is called.

In [74]:
df_test = df_2024_analytics.select(
    "pickup_date",
    "total_amount"
)

In [75]:
df_test.explain()

== Physical Plan ==
*(1) Project [cast(tpep_pickup_datetime#26 as date) AS pickup_date#4379, total_amount#41]
+- *(1) Filter ((((isnotnull(tpep_pickup_datetime#26) AND NOT (((((fare_amount#35 < 0.0) OR (total_amount#41 < 0.0)) OR (tip_amount#38 < 0.0)) OR (tolls_amount#39 < 0.0)) <=> true)) AND NOT (((trip_distance#29 > 100.0) AND CASE WHEN ((cast((unix_timestamp(tpep_dropoff_datetime#27, yyyy-MM-dd HH:mm:ss, Some(America/Tijuana), true) - unix_timestamp(tpep_pickup_datetime#26, yyyy-MM-dd HH:mm:ss, Some(America/Tijuana), true)) as double) / 60.0) > 0.0) THEN ((trip_distance#29 / ((cast((unix_timestamp(tpep_dropoff_datetime#27, yyyy-MM-dd HH:mm:ss, Some(America/Tijuana), true) - unix_timestamp(tpep_pickup_datetime#26, yyyy-MM-dd HH:mm:ss, Some(America/Tijuana), true)) as double) / 60.0) / 60.0)) > 85.0) END) <=> true)) AND (year(cast(tpep_pickup_datetime#26 as date)) = 2024)) AND month(cast(tpep_pickup_datetime#26 as date)) IN (1,2,3))
   +- *(1) ColumnarToRow
      +- FileScan parquet

In [76]:
df_test.explain(True)

== Parsed Logical Plan ==
'Project ['pickup_date, 'total_amount]
+- Project [VendorID#25, pickup_date#4379, pickup_hour#4380, pickup_day_of_week#4381, PULocationID#32, DOLocationID#33, passenger_count#28L, trip_distance#29, trip_duration_minutes#4310, avg_speed_mph#4311, fare_amount#35, tip_amount#38, tolls_amount#39, total_amount#41, payment_type#34L, store_and_fwd_flag#31]
   +- Filter ((((negative_amount_flag#4105 = 0) AND (long_distance_flag#4341 = 0)) AND (year(pickup_date#4379) = 2024)) AND month(pickup_date#4379) IN (1,2,3))
      +- Project [VendorID#25, tpep_pickup_datetime#26, tpep_dropoff_datetime#27, passenger_count#28L, trip_distance#29, RatecodeID#30L, store_and_fwd_flag#31, PULocationID#32, DOLocationID#33, payment_type#34L, fare_amount#35, extra#36, mta_tax#37, tip_amount#38, tolls_amount#39, improvement_surcharge#40, total_amount#41, congestion_surcharge#42, Airport_fee#43, passenger_count_missing#3447, congestion_surcharge_missing#3448, Airport_fee_missing#3449, negat

### Spark Internals: Shuffle

Shuffle happens when Spark needs to redistribute data across partitions.
Operations such as `groupBy`, `join`, `orderBy`, and `distinct` can trigger shuffle.

Shuffle is expensive because it requires data movement between partitions.

In [77]:
df_daily_trips = df_2024_analytics.groupBy(
    "pickup_date"
).count()

In [78]:
df_daily_trips.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[pickup_date#4379], functions=[count(1)])
   +- Exchange hashpartitioning(pickup_date#4379, 12), ENSURE_REQUIREMENTS, [plan_id=2745]
      +- HashAggregate(keys=[pickup_date#4379], functions=[partial_count(1)])
         +- Project [cast(tpep_pickup_datetime#26 as date) AS pickup_date#4379]
            +- Filter ((((isnotnull(tpep_pickup_datetime#26) AND NOT (((((fare_amount#35 < 0.0) OR (total_amount#41 < 0.0)) OR (tip_amount#38 < 0.0)) OR (tolls_amount#39 < 0.0)) <=> true)) AND NOT (((trip_distance#29 > 100.0) AND CASE WHEN ((cast((unix_timestamp(tpep_dropoff_datetime#27, yyyy-MM-dd HH:mm:ss, Some(America/Tijuana), true) - unix_timestamp(tpep_pickup_datetime#26, yyyy-MM-dd HH:mm:ss, Some(America/Tijuana), true)) as double) / 60.0) > 0.0) THEN ((trip_distance#29 / ((cast((unix_timestamp(tpep_dropoff_datetime#27, yyyy-MM-dd HH:mm:ss, Some(America/Tijuana), true) - unix_timestamp(tpep_pickup_datetime#26, yyyy-M

In [79]:
spark.conf.get("spark.sql.shuffle.partitions")

'12'

In [80]:
spark.conf.set("spark.sql.shuffle.partitions", "200")

In [81]:
df_daily_trips = df_2024_analytics.groupBy(
    "pickup_date"
).count()

In [82]:
df_daily_trips.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[pickup_date#4379], functions=[count(1)])
   +- Exchange hashpartitioning(pickup_date#4379, 200), ENSURE_REQUIREMENTS, [plan_id=2762]
      +- HashAggregate(keys=[pickup_date#4379], functions=[partial_count(1)])
         +- Project [cast(tpep_pickup_datetime#26 as date) AS pickup_date#4379]
            +- Filter ((((isnotnull(tpep_pickup_datetime#26) AND NOT (((((fare_amount#35 < 0.0) OR (total_amount#41 < 0.0)) OR (tip_amount#38 < 0.0)) OR (tolls_amount#39 < 0.0)) <=> true)) AND NOT (((trip_distance#29 > 100.0) AND CASE WHEN ((cast((unix_timestamp(tpep_dropoff_datetime#27, yyyy-MM-dd HH:mm:ss, Some(America/Tijuana), true) - unix_timestamp(tpep_pickup_datetime#26, yyyy-MM-dd HH:mm:ss, Some(America/Tijuana), true)) as double) / 60.0) > 0.0) THEN ((trip_distance#29 / ((cast((unix_timestamp(tpep_dropoff_datetime#27, yyyy-MM-dd HH:mm:ss, Some(America/Tijuana), true) - unix_timestamp(tpep_pickup_datetime#26, yyyy-

In [83]:
df_daily_trips.show(5)

+-----------+------+
|pickup_date| count|
+-----------+------+
| 2024-01-07| 66560|
| 2024-01-11|103790|
| 2024-01-02| 74419|
| 2024-01-09| 92816|
| 2024-01-13|103410|
+-----------+------+
only showing top 5 rows


### Broadcast Join Observation

Spark automatically selected BroadcastHashJoin because the taxi zone lookup table is small enough to fit in memory.

The execution plan shows `BroadcastExchange`, which means Spark broadcasted the lookup table instead of shuffling the large trip dataset.

Using broadcast join reduces data movement and improves join performance.

In [84]:
df_join_normal = df_2024_analytics.join(
    df_zones,
    df_2024_analytics.PULocationID == df_zones.LocationID,
    "left"
)

df_join_normal.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [PULocationID#32], [LocationID#3384], LeftOuter, BuildRight, false, false
   :- Project [VendorID#25, cast(tpep_pickup_datetime#26 as date) AS pickup_date#4379, hour(tpep_pickup_datetime#26, Some(America/Tijuana)) AS pickup_hour#4380, dayofweek(cast(tpep_pickup_datetime#26 as date)) AS pickup_day_of_week#4381, PULocationID#32, DOLocationID#33, passenger_count#28L, trip_distance#29, trip_duration_minutes#4310, CASE WHEN (trip_duration_minutes#4310 > 0.0) THEN (trip_distance#29 / (trip_duration_minutes#4310 / 60.0)) END AS avg_speed_mph#4311, fare_amount#35, tip_amount#38, tolls_amount#39, total_amount#41, payment_type#34L, store_and_fwd_flag#31]
   :  +- Project [VendorID#25, tpep_pickup_datetime#26, passenger_count#28L, trip_distance#29, store_and_fwd_flag#31, PULocationID#32, DOLocationID#33, payment_type#34L, fare_amount#35, tip_amount#38, tolls_amount#39, total_amount#41, (cast((unix_timestamp(tpep_dropoff_

In [85]:
df_join_broadcast = df_2024_analytics.join(
    broadcast(df_zones),
    df_2024_analytics.PULocationID == df_zones.LocationID,
    "left"
)

df_join_broadcast.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [PULocationID#32], [LocationID#3384], LeftOuter, BuildRight, false, false
   :- Project [VendorID#25, cast(tpep_pickup_datetime#26 as date) AS pickup_date#4379, hour(tpep_pickup_datetime#26, Some(America/Tijuana)) AS pickup_hour#4380, dayofweek(cast(tpep_pickup_datetime#26 as date)) AS pickup_day_of_week#4381, PULocationID#32, DOLocationID#33, passenger_count#28L, trip_distance#29, trip_duration_minutes#4310, CASE WHEN (trip_duration_minutes#4310 > 0.0) THEN (trip_distance#29 / (trip_duration_minutes#4310 / 60.0)) END AS avg_speed_mph#4311, fare_amount#35, tip_amount#38, tolls_amount#39, total_amount#41, payment_type#34L, store_and_fwd_flag#31]
   :  +- Project [VendorID#25, tpep_pickup_datetime#26, passenger_count#28L, trip_distance#29, store_and_fwd_flag#31, PULocationID#32, DOLocationID#33, payment_type#34L, fare_amount#35, tip_amount#38, tolls_amount#39, total_amount#41, (cast((unix_timestamp(tpep_dropoff_

In [86]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

'10485760b'

In [87]:
df_borough_trips = (
    df_2024_analytics.join(
        F.broadcast(df_zones),
        df_2024_analytics.PULocationID == df_zones.LocationID,
        "left"
    )
    .groupBy("Borough")
    .count()
    .orderBy(F.desc("count"))
)

df_borough_trips.show()

+-------------+-------+
|      Borough|  count|
+-------------+-------+
|    Manhattan|8442776|
|       Queens| 817081|
|     Brooklyn|  95748|
|      Unknown|  31304|
|        Bronx|  24723|
|          N/A|   4916|
|          EWR|    915|
|Staten Island|    220|
+-------------+-------+



### Broadcast Join Example

The taxi trip dataset was joined with the taxi zone lookup table using a broadcast join.

This allowed the analytical dataset to include borough names instead of only location IDs.

The joined dataset was then aggregated to calculate the total number of trips by borough.

In [88]:
df_borough_trips.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (14)
+- Sort (13)
   +- Exchange (12)
      +- HashAggregate (11)
         +- Exchange (10)
            +- HashAggregate (9)
               +- Project (8)
                  +- BroadcastHashJoin LeftOuter BuildRight (7)
                     :- Project (3)
                     :  +- Filter (2)
                     :     +- Scan parquet  (1)
                     +- BroadcastExchange (6)
                        +- Filter (5)
                           +- Scan csv  (4)


(1) Scan parquet 
Output [8]: [tpep_pickup_datetime#26, tpep_dropoff_datetime#27, trip_distance#29, PULocationID#32, fare_amount#35, tip_amount#38, tolls_amount#39, total_amount#41]
Batched: true
Location: InMemoryFileIndex [file:/C:/Users/Murch24/aws-pyspark-data-engineering-project/data/raw/yellow_tripdata_2024-01.parquet, ... 2 entries]
PushedFilters: [IsNotNull(tpep_pickup_datetime)]
ReadSchema: struct<tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,trip_dista

In [89]:
df_2024_analytics.groupBy("PULocationID") \
    .count() \
    .orderBy(
        F.desc("count")
    ) \
    .show(10)

+------------+------+
|PULocationID| count|
+------------+------+
|         161|447802|
|         237|434215|
|         132|416440|
|         236|412391|
|         162|332218|
|         230|325686|
|         186|315496|
|         142|312345|
|         138|281280|
|         239|278475|
+------------+------+
only showing top 10 rows


In [90]:
df_2024_analytics.explain()

== Physical Plan ==
*(1) Project [VendorID#25, cast(tpep_pickup_datetime#26 as date) AS pickup_date#4379, hour(tpep_pickup_datetime#26, Some(America/Tijuana)) AS pickup_hour#4380, dayofweek(cast(tpep_pickup_datetime#26 as date)) AS pickup_day_of_week#4381, PULocationID#32, DOLocationID#33, passenger_count#28L, trip_distance#29, trip_duration_minutes#4310, CASE WHEN (trip_duration_minutes#4310 > 0.0) THEN (trip_distance#29 / (trip_duration_minutes#4310 / 60.0)) END AS avg_speed_mph#4311, fare_amount#35, tip_amount#38, tolls_amount#39, total_amount#41, payment_type#34L, store_and_fwd_flag#31]
+- *(1) Project [VendorID#25, tpep_pickup_datetime#26, passenger_count#28L, trip_distance#29, store_and_fwd_flag#31, PULocationID#32, DOLocationID#33, payment_type#34L, fare_amount#35, tip_amount#38, tolls_amount#39, total_amount#41, (cast((unix_timestamp(tpep_dropoff_datetime#27, yyyy-MM-dd HH:mm:ss, Some(America/Tijuana), true) - unix_timestamp(tpep_pickup_datetime#26, yyyy-MM-dd HH:mm:ss, Some(Am

In [91]:
df_test = df_2024_analytics.groupBy("pickup_date").count()

In [92]:
df_test.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Project (3)
            +- Filter (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [7]: [tpep_pickup_datetime#26, tpep_dropoff_datetime#27, trip_distance#29, fare_amount#35, tip_amount#38, tolls_amount#39, total_amount#41]
Batched: true
Location: InMemoryFileIndex [file:/C:/Users/Murch24/aws-pyspark-data-engineering-project/data/raw/yellow_tripdata_2024-01.parquet, ... 2 entries]
PushedFilters: [IsNotNull(tpep_pickup_datetime)]
ReadSchema: struct<tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,trip_distance:double,fare_amount:double,tip_amount:double,tolls_amount:double,total_amount:double>

(2) Filter
Input [7]: [tpep_pickup_datetime#26, tpep_dropoff_datetime#27, trip_distance#29, fare_amount#35, tip_amount#38, tolls_amount#39, total_amount#41]
Condition : ((((isnotnull(tpep_pickup_datetime#26) AND NOT (((((fare_amount#35 < 0

In [93]:
df_2024_analytics.limit(5).show()

+--------+-----------+-----------+------------------+------------+------------+---------------+-------------+---------------------+------------------+-----------+----------+------------+------------+------------+------------------+
|VendorID|pickup_date|pickup_hour|pickup_day_of_week|PULocationID|DOLocationID|passenger_count|trip_distance|trip_duration_minutes|     avg_speed_mph|fare_amount|tip_amount|tolls_amount|total_amount|payment_type|store_and_fwd_flag|
+--------+-----------+-----------+------------------+------------+------------+---------------+-------------+---------------------+------------------+-----------+----------+------------+------------+------------+------------------+
|       2| 2024-01-01|          0|                 2|         186|          79|              1|         1.72|                 19.8| 5.212121212121212|       17.7|       0.0|         0.0|        22.7|           2|                 N|
|       1| 2024-01-01|          0|                 2|         140|      

In [94]:
output_path = r"C:\Users\Murch24\aws-pyspark-data-engineering-project\data\processed\yellow_taxi_2024"

(
    df_2024_analytics
    .withColumn("month", F.month("pickup_date"))
    .write
    .mode("overwrite")
    .partitionBy("month")
    .parquet(output_path)
)

In [95]:
output_path = r"C:\Users\Murch24\aws-pyspark-data-engineering-project\data\processed\yellow_taxi_2024"

(
    df_2024_analytics
    .withColumn("month", F.month("pickup_date"))
    .coalesce(4)
    .write
    .mode("overwrite")
    .partitionBy("month")
    .parquet(output_path)
)

In [96]:
df_2024_analytics.select(
    F.month("pickup_date").alias("month")
).groupBy("month").count().orderBy("month").show()

+-----+-------+
|month|  count|
+-----+-------+
|    1|2927019|
|    2|2966758|
|    3|3523906|
+-----+-------+



In [97]:
df_2024_analytics.select(
    F.min("pickup_date"),
    F.max("pickup_date")
).show()

+----------------+----------------+
|min(pickup_date)|max(pickup_date)|
+----------------+----------------+
|      2024-01-01|      2024-03-31|
+----------------+----------------+



In [98]:
output_path = r"C:\Users\Murch24\aws-pyspark-data-engineering-project\data\processed\yellow_taxi_2024"

months = (
    df_2024_analytics
    .select(F.month("pickup_date").alias("month"))
    .distinct()
    .collect()
)

for row in months:
    month = row["month"]

    (
        df_2024_analytics
        .filter(F.month("pickup_date") == month)
        .coalesce(1)
        .write
        .mode("overwrite")
        .parquet(f"{output_path}\\month={month}")
    )

## Parquet File Size and Partition Analysis

After writing partitioned Parquet data, I analyze the output structure to check:

- Number of Parquet files created in each partition
- Total storage size of each partition
- Average file size

This helps evaluate the write strategy and avoid:
- too many small files
- excessively large files
- inefficient partitioning

In [101]:
import os
for root, dirs, files in os.walk(output_path):
    parquet_files = [f for f in files if f.endswith(".parquet")]
    
    if parquet_files:
        total_size = sum(
            os.path.getsize(os.path.join(root, f))
            for f in parquet_files
        )
        print(
            root,
            "files:", len(parquet_files),
            "size MB:", round(total_size / (1024**2), 2)
        )

C:\Users\Murch24\aws-pyspark-data-engineering-project\data\processed\yellow_taxi_2024\month=1 files: 1 size MB: 51.57
C:\Users\Murch24\aws-pyspark-data-engineering-project\data\processed\yellow_taxi_2024\month=2 files: 1 size MB: 52.15
C:\Users\Murch24\aws-pyspark-data-engineering-project\data\processed\yellow_taxi_2024\month=3 files: 1 size MB: 62.22


## Read Data from a Specific Partition

Test partition pruning by loading only February data from the partitioned Parquet dataset.

In [102]:
df_partition_pruning = (
    spark.read.parquet(output_path)
    .filter(F.col("month").isin([2, 3]))
)

In [103]:
df_partition_pruning.select(
    F.min("pickup_date"),
    F.max("pickup_date")
).show()

+----------------+----------------+
|min(pickup_date)|max(pickup_date)|
+----------------+----------------+
|      2024-02-01|      2024-03-31|
+----------------+----------------+



In [104]:
output_path = r"C:\Users\Murch24\aws-pyspark-data-engineering-project\data\processed\yellow_taxi_2024_2"
df_analytics_month_year = df_2024_analytics
(
    df_analytics_month_year
    .withColumn("month", F.month("pickup_date")) \
    .withColumn("year", F.year("pickup_date"))     
    .write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet(output_path)
)


In [105]:
for root, dirs, files in os.walk(output_path):
    parquet_files = [f for f in files if f.endswith(".parquet")]
    
    if parquet_files:
        total_size = sum(
            os.path.getsize(os.path.join(root, f))
            for f in parquet_files
        )
        print(
            root,
            "files:", len(parquet_files),
            "size MB:", round(total_size / (1024**2), 2)
        )

C:\Users\Murch24\aws-pyspark-data-engineering-project\data\processed\yellow_taxi_2024_2\year=2024\month=1 files: 4 size MB: 51.05
C:\Users\Murch24\aws-pyspark-data-engineering-project\data\processed\yellow_taxi_2024_2\year=2024\month=2 files: 5 size MB: 51.69
C:\Users\Murch24\aws-pyspark-data-engineering-project\data\processed\yellow_taxi_2024_2\year=2024\month=3 files: 4 size MB: 61.32


## Partition Pruning and Predicate Pushdown Validation

Test Spark optimization when reading partitioned Parquet data.

- **Partition pruning** allows Spark to read only the required partitions (for example, `year=2024/month=2`) instead of scanning all folders.
- **Predicate pushdown** allows Spark to push filters to the Parquet reader and skip unnecessary data inside the files.

The query below combines both optimizations by filtering:
- partition columns: `year` and `month`
- data column: `PULocationID`

In [106]:
df_feb_2024 = (
    spark.read.parquet(output_path)
    .filter((F.col("month") == 2) & (F.col("year") == 2024) &  (F.col("PULocationID").isin([132, 1, 138]))
    )
)

In [ ]:
spark.read.parquet(output_path).printSchema()